# Assignment 07: Critique, Redesign, and Explain Cardiac Rehab Charts

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [ ]:
import json
from pathlib import Path

import altair as alt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("Altair:", alt.__version__)

DATA_DIR = Path("data")
PATIENTS_PATH = DATA_DIR / "rehab_patients.csv"
ATTENDANCE_PATH = DATA_DIR / "session_attendance.csv"
FOLLOWUP_PATH = DATA_DIR / "followup_goals.csv"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
EXPLORATORY_SPEC_PATH = OUTPUT_DIR / "exploratory_spec.json"
EVIDENCE_PATH = OUTPUT_DIR / "visualization_evidence.json"
REDESIGN_PATH = OUTPUT_DIR / "critique_redesign.png"
SUPPORTING_DATA_PATH = OUTPUT_DIR / "explanatory_supporting_data.csv"
EXPLANATORY_PATH = OUTPUT_DIR / "explanatory_chart.png"
TEXT_ALTERNATIVE_PATH = OUTPUT_DIR / "explanatory_text_alternative.txt"

BLUE = "#0072B2"  # colorblind-safe blue
ORANGE = "#E69F00"  # colorblind-safe orange

print("data files found:", PATIENTS_PATH.exists() and ATTENDANCE_PATH.exists() and FOLLOWUP_PATH.exists())

## Task 1: Explore the rehab patients with Altair

### 1.1 Load the patients and build the scatter plot

> **Checkpoint: `output/exploratory_spec.json`**
> A Vega-Lite specification with a `point` mark, `x`, `y`, `color`, and `shape` encodings, and the twelve patients embedded.

In [ ]:
patients = pd.read_csv(PATIENTS_PATH)  # TODO: read PATIENTS_PATH with pd.read_csv

# TODO: print the shape and the dtypes of patients
print(patients.shape)
print(patients.dtypes)

# TODO: the Altair scatter plot README Task 1.1 describes
exploratory_chart = alt.Chart(patients).mark_point(filled=True, size=90).encode(
    x = alt.X('sessions_attended:Q', title='Sessions attended (out of 36)'),
    y = alt.X('walk_distance_m:Q', title='Six-minute walk distance (m)', scale=alt.Scale(zero=False)),
    color = alt.Color("program:N"),
    shape = alt.Shape("program:N"),
    tooltip = ['sessions_attended:Q', 'walk_distance_n:Q', 'program:N']
).properties(title='Six minute walking distance (m) vs. Walking sessions attended')  

# TODO: save exploratory_chart to EXPLORATORY_SPEC_PATH with .save()

exploratory_chart.save(EXPLORATORY_SPEC_PATH)
display(exploratory_chart)
display(patients)

### 1.2 Describe what the chart shows

Responses:

This chart explores the association between hosting walking sessions and physical activity (total walking distance over six-minute sessions). One point represents the total walking distance (y-axis) for a given patient that attended a certain number of walking sessions (x-axis). A blue circular point represents a patient who attended center-based sessions and an orange square point represents a patient who attended home-based sessions.

Based on the graph, both center-based and home-based programs experienced a general increase in six-minute walking distance as the number of walking sessions increased. As number of walking sessions increased, center-based programs achieved a higher total walking distance than home-based programs. Twelve patients alone can't show the relationships between attending more sessions and walking distance as they may not be a representative sample, as individuals may be healthier and able to walk longer distances than others, introducing confounding to the association.

## Task 2: Critique and redesign a misleading chart

### 2.1 Inspect the draft chart

The next cell draws a draft chart from the rehab service's dashboard. Run it and look at the chart before you start Task 2.2.

In [ ]:
attendance = pd.read_csv(ATTENDANCE_PATH)
display(attendance)

quarters = ["Q1", "Q2"]
x = np.arange(len(quarters))  # [0 1]: one position per quarter
width = 0.38
home = attendance.loc[attendance["program"] == "Home-based", "attended_pct"]
center = attendance.loc[attendance["program"] == "Center-based", "attended_pct"]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width / 2, home, width, label="Home-based", color="red",
       hatch="xx", edgecolor="black", linewidth=2)
ax.bar(x + width / 2, center, width, label="Center-based", color="green",
       hatch="xx", edgecolor="black", linewidth=2)
ax.set_xticks(x, quarters)
ax.set_ylim(76, 83)
ax.grid(True, linewidth=2, color="#555555")
ax.set_title("Center-based rehab caused better attendance!")
ax.legend()
plt.show()

### 2.2 Write the critique

Fill in a `problem` and a `repair` for each category, then run the cell to save the critique.

> **Checkpoint: `output/visualization_evidence.json`**
> An object with the key `critique`: a list of five entries, each with the keys `category`, `problem`, and `repair`.

In [ ]:
critique = [
    {"category": "unsupported claim", "problem": "The title of the bar chart displays a unsupported claim that can't be stated without further analysis.", "repair": "Edited the title to state the association between program and attendance, without making further claims."},  # TODO: fill in each problem and repair
    {"category": "truncated baseline", "problem": "The y-axis starts at 76 instead of 0.", "repair": "Adjusting the y-axis values so that it starts at 0."},
    {"category": "missing unit", "problem": "The y-axis is missing its label/unit of measurement.", "repair": "Labeled y-axis with 'percent'."},
    {"category": "color-only encoding", "problem": "Chart uses red and green to distinguish the two programs, which may be difficult to read for red-green color-blind users.", "repair": "Adjusted the color scheme to show different colors."},
    {"category": "distracting decoration", "problem": "The chart contains extra edges, hatches, and grid lines that are distracting and don't add anything to the data.", "repair": "Removed the hatches and grid lines from chart."},
]

for entry in critique:
    print(f"{entry['category']}: {entry['problem']}")
    print(f"    repair: {entry['repair']}")

with open(EVIDENCE_PATH, "w", encoding="utf-8") as file:
    json.dump({"critique": critique}, file, indent=2, ensure_ascii=False)

### 2.3 Redesign the chart

> **Checkpoint: `output/critique_redesign.png`**
> Your redesigned chart as a PNG image.

In [ ]:
redesign_fig, redesign_ax = plt.subplots(figsize=(7, 4.5))

# TODO: draw the four bars again from home and center on redesign_ax, repairing each problem in your critique
# (README Task 2.3 lists the repairs)
redesign_ax.set_ylim(0, 100)
redesign_ax.set_xticks(x, quarters)
redesign_ax.set(xlabel = "Quarter", ylabel = "Scheduled sessions attended (%)")
home_1 = redesign_ax.bar(x - width / 2, home, width, label="Home-based", color="ORANGE", hatch="..", linewidth=2)
center_1 = redesign_ax.bar(x + width / 2, center, width, label="Center-based", color="BLUE", hatch="//", linewidth=2)
home_labels = redesign_ax.bar_label(home_1, fmt='%d%%')
center_labels = redesign_ax.bar_label(center_1, fmt='%d%%')
redesign_ax.set(title= "Six-minute walking distance for home-based and center-based programs by quarter")
redesign_ax.spines[['top','right']].set_visible(False)
redesign_ax.legend(title='Program', loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

# TODO: save redesign_fig to REDESIGN_PATH with dpi=150 and bbox_inches="tight"
redesign_fig.savefig(REDESIGN_PATH, dpi=150, bbox_inches="tight")


plt.show()
bottom, top = redesign_ax.get_ylim()
print("y-axis runs from", bottom, "to", top)


## Task 3: Explain one finding to the program coordinator

### 3.1 Write the contract and save the supporting data

> **Checkpoint: `output/explanatory_supporting_data.csv`**
> The header line `program,visit_number,goal_met_pct`, then the eight rows of `data/followup_goals.csv`.

In [ ]:
followup = pd.read_csv(FOLLOWUP_PATH)  # TODO: read FOLLOWUP_PATH with pd.read_csv
# TODO: display followup
display(followup)

question = "How did the percent of patients meeting rehab goal differ between the home-based and center-based walking programs?"  # TODO: the comparison the chart should help the coordinator make
audience = "Rehab program coordinators; they would utilize this chart to evaluate the effectiveness of each program and which one to provide more follow-up support in the future"  # TODO: who reads the chart and what they will use it for
intended_claim = "Center-based rehab programs have a higher % of patients meeting goal than home-based rehab programs."  # TODO: the one descriptive conclusion the chart supports
y_measure = "Y-axis measures the percent of patients meeting rehab goal (%) for a program."  # TODO: what the y-axis measures, with its unit
grain = "The percent of patients meeting rehab goals in a given visit"  # TODO: what one row of the plotting table and one point on a line represent
data_types = {
    "program": "categorical", 
    "visit_number": "ordinal",
    "goal_met_pct": "quantitative"
    }

supporting_data = pd.DataFrame({"program": followup["program"], "visit_number": followup["visit_number"], "goal_met_pct": followup["goal_met_pct"]})  # TODO: the three plotted columns of followup

# TODO: save supporting_data to SUPPORTING_DATA_PATH without the row index
supporting_data.to_csv(SUPPORTING_DATA_PATH, index=False)

round_trip = pd.read_csv(SUPPORTING_DATA_PATH)
display(supporting_data)
print("shape:", round_trip.shape)
print("same as saved:", round_trip.equals(supporting_data))

### 3.2 Draw the explanatory chart

> **Checkpoint: `output/explanatory_chart.png`**
> Your explanatory line chart as a PNG image.

In [ ]:
home_goals = supporting_data.loc[supporting_data["program"] == "Home-based"]
center_goals = supporting_data.loc[supporting_data["program"] == "Center-based"]

explanatory_fig, explanatory_ax = plt.subplots(figsize=(8, 4.8))

# TODO: draw one line per program on explanatory_ax, with visit_number on x and goal_met_pct on y,
# and finish the chart as README Task 3.2 describes
explanatory_ax.plot(home_goals["visit_number"], home_goals["goal_met_pct"], 's--', color='ORANGE')
explanatory_ax.plot(center_goals["visit_number"], center_goals["goal_met_pct"], 'o-', color='BLUE')
explanatory_ax.annotate("Center-based visit-4 point, Gap at visit 4: 79% against 70%, 9 points", xy=(4,79), xytext=(1, 79), arrowprops=dict(arrowstyle='->'))
explanatory_ax.set(xlabel="Follow-up visit", ylabel="Patients meeting the exercise goal (%)", title="Increase in % patients meeting rehab goal over follow-up visits")
visits = [1,2,3,4]
explanatory_ax.set_xticks(visits)
explanatory_ax.spines[['top','right']].set_visible(False)

# TODO: save explanatory_fig to EXPLANATORY_PATH with dpi=150 and bbox_inches="tight"
explanatory_fig.savefig(EXPLANATORY_PATH, dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Write the text alternative and save the evidence

> **Checkpoint: `output/visualization_evidence.json`**
> The keys `critique`, `question`, `audience`, `intended_claim`, `y_measure`, `grain`, `data_types`, and `text_alternative`.

> **Checkpoint: `output/explanatory_text_alternative.txt`**
> The same paragraph as `text_alternative` in the JSON.

In [ ]:
text_alternative = ('Task 3.2 illustrates a line graph. X-axis displays follow-up visit with the units 1, 2, 3, or 4. The y-axis describes the percent of patients meeting rehab goal by each program, with units being %. Both home-based and center-based programs experience a general increase in % patients meeting rehab goal from visit 1 to visit 4. The gap between center-based and home-based programs is 9% points (79% vs. 70%).'
                    'Limitation: Each percentage only represents patients who were present at a visit, so the 9% gap between center-based and home-based programs does not take into account individuals who were lost-to-follow-up and did not return at visit 4.')

visualization_evidence = {
    "critique": critique,
    "question": question,
    "audience": audience,
    "intended_claim": intended_claim,
    "y_measure": y_measure,
    "grain": grain,
    "data_types": data_types,
    "text_alternative": text_alternative
}

# TODO: save visualization_evidence to EVIDENCE_PATH with json.dump, as the Task 2.2 cell does
with open(EVIDENCE_PATH, "w", encoding="utf-8") as file:
    json.dump(visualization_evidence, file, indent=2, ensure_ascii=False)


# TODO: write text_alternative to TEXT_ALTERNATIVE_PATH
with open(TEXT_ALTERNATIVE_PATH, "w", encoding="utf-8") as file:
    file.write(text_alternative)


with open(EVIDENCE_PATH, encoding="utf-8") as file:
    saved = json.load(file)
print(list(saved))

### 3.4 Review the saved charts

Open `output/critique_redesign.png` and `output/explanatory_chart.png`, and answer each question with what you see in them.

1. Printed in grayscale, how would a reader still tell the two programs apart in each chart?
2. Where does the explanatory chart's annotation point, and does its wording stay within what these eight rows can show?
3. Is any text cut off or overlapping in either saved PNG? If so, what did you change?
4. What limitation remains that no chart design can fix?

Responses:
1. A reader can still tell the two programs apart as critique_redesign.png fills in the bars differently (dots for home-based and slashes for center-based). explanatory_chart.png also draws the data points differently (circles and solid lines for center-based, squares and dashed lines for home-based).

2. The annotation points to the percent goals met for the center-based program at visit 4. Yes, the wording stays within what the eight rows shows as it
doesn't make any claims and remains objective by stating the difference between the center-based and home-based programs at visit 4.

3. There is no text cut off or overlapping in either PNG. This format was achieving by placing the legend on the right edge of the graph in critique_redesign.png and by placing the annotation on the left side of the data point in explanatory_chart.png so that it doesn't trail off outside of the graph.

4. Both charts describe overall data at the program/group level rather than the individual level. As a result, the charts do not illustrate specific patient characteristics that may affect goals met including their age, gender, and ability to do physical activity.

## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [ ]:
from check_assignment import run_checks

_ = run_checks()